In [1]:
import pandas as pd
import numpy as np
import glob, os, shutil, cv2, json, sys
from tqdm import tqdm

In [2]:
def getFiles(path, limit=None, shuffle=False):
    target = sorted(glob.glob(os.path.join(path, '*')))
    if shuffle:
        np.random.shuffle(target) 
    return target[:limit]


OPTIONS = json.loads(open('../Task/info.json', 'r').read())
OPTIONS

{'network': 'resacunet2',
 'img_size': [16, 128, 128],
 'lr': 0.0005,
 'loss': 'dice_focal',
 'batch_size': 2,
 'scheduler': 'plateau',
 'dropout': 0.1,
 'num_filters': 16}

In [3]:
df = pd.DataFrame()
df['img_path']  = [os.path.join('../Dataset', path) for path in getFiles('images')]
df['mask_path'] = [os.path.join('../Dataset', path) for path in getFiles('masks')]
df['shape']  = [np.load(path).shape for path in getFiles('images')]
df

,img_path,mask_path,shape
0,../Dataset/images/0.npy,../Dataset/masks/0.npy,"(128, 128, 128)"
1,../Dataset/images/1.npy,../Dataset/masks/1.npy,"(128, 128, 128)"
2,../Dataset/images/10.npy,../Dataset/masks/10.npy,"(128, 128, 128)"
3,../Dataset/images/100.npy,../Dataset/masks/100.npy,"(128, 128, 128)"
4,../Dataset/images/101.npy,../Dataset/masks/101.npy,"(128, 128, 128)"
...,...,...,...
215,../Dataset/images/95.npy,../Dataset/masks/95.npy,"(128, 128, 128)"
216,../Dataset/images/96.npy,../Dataset/masks/96.npy,"(128, 128, 128)"
217,../Dataset/images/97.npy,../Dataset/masks/97.npy,"(128, 128, 128)"
218,../Dataset/images/98.npy,../Dataset/masks/98.npy,"(128, 128, 128)"


In [4]:
if OPTIONS.get('img_size') is None:
    df.to_csv('DataBase.csv', index=None)
    display(df)
    sys.exit("Finalizando o programa: sem tiles nessa rodada")

# TILES

In [5]:
IMG_SIZE = tuple(OPTIONS['img_size'])
IMG_SIZE

(16, 128, 128)

In [6]:
class TilesBuilder:
    def __init__(self, target_size=(8, 64, 128), main_dir='tiles'):
        self.target_size = target_size
        self.main_dir    = main_dir

        if os.path.exists(main_dir):
            shutil.rmtree(main_dir)
        os.makedirs(main_dir)

    def update(self, file_path, target_dir, is_mask=False):
        folder = os.path.join(self.main_dir, target_dir)
        os.makedirs(folder, exist_ok=True)

        data = np.load(file_path)
        
        # ---------------------------------------------------------
        # CORREÇÃO DE ROTAÇÃO AQUI
        # Se o dado vem como (64, 128, 4) e precisa virar (4, 64, 128)
        # O eixo que está no índice 2 (tamanho 4) vai para o índice 0.
        # Os outros "escorregam" para a direita.
        # ---------------------------------------------------------
        data = np.transpose(data, (2, 0, 1)) 
        
        # Caso precise inverter também o X e o Y (ex: virar 4, 128, 64), 
        # a ordem do transpose seria (2, 1, 0).
        
        base_name     = os.path.splitext(os.path.basename(file_path))[0]
        z_t, y_t, x_t = self.target_size
        
        z_pad = (z_t - data.shape[0] % z_t) % z_t
        y_pad = (y_t - data.shape[1] % y_t) % y_t
        x_pad = (x_t - data.shape[2] % x_t) % x_t

        pad_mode    = 'constant' if is_mask else 'reflect'
        data_padded = np.pad(data, ((0, z_pad), (0, y_pad), (0, x_pad)), mode=pad_mode) if z_pad > 0 or y_pad > 0 or x_pad > 0 else data
            
        index = 0
        for z in range(0, data_padded.shape[0], z_t):
            for y in range(0, data_padded.shape[1], y_t):
                for x in range(0, data_padded.shape[2], x_t):
                    tile = data_padded[z:z+z_t, y:y+y_t, x:x+x_t]
                    save_path = os.path.join(folder, f'{base_name}_tile_{index}.npy')
                    
                    np.save(save_path, tile)
                    index = (index + 1)


tiles = TilesBuilder(target_size=IMG_SIZE, main_dir='tiles')

for img_path, mask_path in tqdm(zip(df.img_path, df.mask_path), total=len(df)):
    tiles.update(img_path,  'images/', is_mask=False)
    tiles.update(mask_path, 'masks/',  is_mask=True)

100%|██████████| 220/220 [00:14<00:00, 15.40it/s]


In [7]:
print(len(getFiles('tiles/images')), 'imagens geradas')

1760 imagens geradas


In [8]:
df = pd.DataFrame()
df['img_path']  = [os.path.join('../Dataset', path) for path in getFiles('tiles/images')]
df['mask_path'] = [os.path.join('../Dataset', path) for path in getFiles('tiles/masks')]
df['shape'] = [tiles.target_size for _ in range(len(df))]

df.to_csv('DataBase.csv', index=None)
df

,img_path,mask_path,shape
0,../Dataset/tiles/images/0_tile_0.npy,../Dataset/tiles/masks/0_tile_0.npy,"(16, 128, 128)"
1,../Dataset/tiles/images/0_tile_1.npy,../Dataset/tiles/masks/0_tile_1.npy,"(16, 128, 128)"
2,../Dataset/tiles/images/0_tile_2.npy,../Dataset/tiles/masks/0_tile_2.npy,"(16, 128, 128)"
3,../Dataset/tiles/images/0_tile_3.npy,../Dataset/tiles/masks/0_tile_3.npy,"(16, 128, 128)"
4,../Dataset/tiles/images/0_tile_4.npy,../Dataset/tiles/masks/0_tile_4.npy,"(16, 128, 128)"
...,...,...,...
1755,../Dataset/tiles/images/9_tile_3.npy,../Dataset/tiles/masks/9_tile_3.npy,"(16, 128, 128)"
1756,../Dataset/tiles/images/9_tile_4.npy,../Dataset/tiles/masks/9_tile_4.npy,"(16, 128, 128)"
1757,../Dataset/tiles/images/9_tile_5.npy,../Dataset/tiles/masks/9_tile_5.npy,"(16, 128, 128)"
1758,../Dataset/tiles/images/9_tile_6.npy,../Dataset/tiles/masks/9_tile_6.npy,"(16, 128, 128)"
